** Cattle Detection: Aerial Monitoring Using YOLOv5 **

**1. Cloning The YOLOv5 repository (Do this part only once)**

In [ ]:
#clone YOLOv5
!git clone https://github.com/ultralytics/yolov5

# Change the current working directory.
%cd yolov5 
%pip install -qr requirements.txt # install dependencies from requirements to use Yolov5 with quiet option

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**2. Import Required Libraries**

In [ ]:
import os
import shutil
import cv2
import numpy as np
import torch
import yaml
from IPython.display import Image, display

**3. Loading the dataset (Do this part only once)**


In [ ]:
# source path
raw_train_img = "../raw_dataset/train/images"
raw_train_label = "../raw_dataset/train/labels"
raw_val_img = "../raw_dataset/valid/images"
raw_val_label = "../raw_dataset/valid/labels"
raw_test_img = "../raw_dataset/test/images"
raw_test_label = "../raw_dataset/test/labels"

In [ ]:
# Destination path
train_image_path = "../datasets2/images/train"
train_label_path = "../datasets2/labels/train"
val_image_path = "../datasets2/images/val"
val_label_path = "../datasets2/labels/val"
test_image_path = "../datasets2/images/test"
test_label_path = "../datasets2/labels/test"

In [ ]:
# copy files from source directory into dest directory
shutil.copytree(raw_train_img, train_image_path)
shutil.copytree(raw_train_label, train_label_path)
shutil.copytree(raw_val_img, val_image_path)
shutil.copytree(raw_val_label, val_label_path)

In [ ]:
shutil.copytree(raw_test_img, test_image_path)
shutil.copytree(raw_test_label, test_label_path)

**4-Check dataset**

In [ ]:
# Checking the size of images and displaying them
import numpy as np
import cv2

# Image shape in Training
image = cv2.imread('../datasets2/images/train/4818b4db76e1a51e2743ac9ca5b1eae7_jpg.rf.4bfa0d1602b206b40b19fc36d5fb2d81.jpg')
height = np.size(image, 0)
width = np.size(image, 1)
print ("shape of the training image {}, {}".format(height, width))

# Image shape in validation
image = cv2.imread('../datasets2/images/val/DJI_0064_JPG_jpg.rf.8806c859c43f8960106e68ee22642947.jpg')
height = np.size(image, 0)
width = np.size(image, 1)
print ("shape of the validation image {}, {}".format(height, width))

In [ ]:
# dispying with different width
Image(filename='../datasets2/images/train/4818b4db76e1a51e2743ac9ca5b1eae7_jpg.rf.4bfa0d1602b206b40b19fc36d5fb2d81.jpg', width=416) 

In [ ]:
# dispying with different width
Image(filename='../datasets2/images/val/DJI_0064_JPG_jpg.rf.8806c859c43f8960106e68ee22642947.jpg', width=416) 

**5. Create cow_data.yaml (dataset config file)**

`data/cow_data.yaml`, created below, is the dataset configuration file that defines:

1) The dataset root directory path and relative paths to train, val, and test image directories.

2) nc: The number of classes (1).

3) names: A list of class names (['cow']).

In [ ]:
import yaml

result = {
    "train": f"../datasets2/images/train", # train images 
    "val": f"../datasets2/images/val",     # val images 
    "nc": 1,                              # number of classes
    "names": ['cow'],  # class names
}
with open("./data/cow_data.yaml", "w") as f:
    dump = yaml.dump(result, default_flow_style=False)
    f.write(dump)

**6. Training Our Custom Cattle Detector Model**

Train a YOLOv5s model on the aerial cattle dataset by specifying the dataset configuration, batch size, image resolution, and pretrained weights (`yolov5s.pt`). Pretrained weights are automatically downloaded from the official YOLOv5 release for transfer learning.

Parameters:
- `--data`: Path to the dataset configuration file (`data/cow_data.yaml`).
- `--weights`: Initial weights for transfer learning (`yolov5s.pt`).
- `--img`: Input image resolution (608x608).
- `--batch`: Batch size for gradient updates.
- `--epochs`: Total number of training epochs (30).
- `--cache`: Cache images in RAM for faster training throughput.

In [ ]:
# Training YOLOv5s on cattle dataset for 30 epochs
!python train.py --img 608 --batch 9 --epochs 30 --data data/cow_data.yaml --weights yolov5s.pt --cache

All training results are saved to runs/train/ with incrementing run directories, i.e. runs/train/exp2, runs/train/exp3 etc.

**7. Local Logging:**

All training artifacts and evaluation metrics are automatically logged to `runs/train/exp/`. This includes training mosaics, validation batch ground-truth labels, prediction overlays, and augmentation visualizations. 

Below displays the validation batch ground-truth labels (`val_batch1_labels.jpg`):

In [ ]:
# Display validation batch ground-truth labels
Image(filename='./runs/train/exp/val_batch1_labels.jpg', width=800)

Training losses, precision, recall, and mAP metrics are automatically logged to a `results.csv` file and rendered as `results.png` upon training completion:

In [ ]:
Image(filename='./runs/train/exp/results.png', width=800)

In [ ]:
Image(filename='./runs/train/exp/val_batch0_pred.jpg', width=800)

**8. Run Inference With Custom YOLOv5 Object Detector Trained Weights**

Execute `detect.py` to run model inference on the test dataset and video inputs:

- `--source`: Path to the input file (image or video) or directory for inference.
- `--weights`: Path to the trained model checkpoint (`runs/train/exp/weights/best.pt`).
- `--conf`: Confidence threshold for filtering valid detections.
- `--save-txt`: Saves detected bounding box coordinates and class predictions to text files.

**Inference on Images:**

In [ ]:
!python detect.py --source ../datasets2/images/test/6bfad02b487e438d3e3698ff3d0c27bb_jpg.rf.037882a01d90fcb501d1fbc95ee8d1a2.jpg --weights runs/train/exp/weights/best.pt --conf 0.5 --save-txt

In [ ]:
Image(filename='./runs/detect/exp/6bfad02b487e438d3e3698ff3d0c27bb_jpg.rf.037882a01d90fcb501d1fbc95ee8d1a2.jpg', width=800)

In [ ]:
!python detect.py --source ../datasets2/images/test/DJI_0063_JPG_jpg.rf.6dc91f01f76d2ea1d646586eddfa69a7.jpg --weights runs/train/exp/weights/best.pt --conf 0.5 --save-txt

In [ ]:
Image(filename='./runs/detect/exp2/DJI_0063_JPG_jpg.rf.6dc91f01f76d2ea1d646586eddfa69a7.jpg', width=800)

In [ ]:
# dispying with different width
!python detect.py --source ../datasets2/images/test/a80f679b64a3c523f96ef4bfb31b9171_jpg.rf.4edc06e17efb7ad6d7a0f53e4d337aa4.jpg --weights runs/train/exp/weights/best.pt --conf 0.5 --save-txt

In [ ]:
Image(filename='./runs/detect/exp3/a80f679b64a3c523f96ef4bfb31b9171_jpg.rf.4edc06e17efb7ad6d7a0f53e4d337aa4.jpg', width=800)

In [ ]:
!python detect.py --source ../datasets2/images/test/d48a359c5c1c28f13d5c1b6cfe5f7652_jpg.rf.81942c0d4bb548f07f86a1813eaa6338.jpg --weights runs/train/exp/weights/best.pt --conf 0.5 --save-txt

In [ ]:
Image(filename='./runs/detect/exp4/d48a359c5c1c28f13d5c1b6cfe5f7652_jpg.rf.81942c0d4bb548f07f86a1813eaa6338.jpg', width=800)

In [ ]:
!python detect.py --source ../datasets2/images/test/1a1c66d4ee9f6dbb059a84633d59ba38_jpg.rf.7e593dc121ec6da5b85d4945ce752bc8.jpg --weights runs/train/exp/weights/best.pt --conf 0.5 --save-txt

In [ ]:
Image(filename='./runs/detect/exp5/1a1c66d4ee9f6dbb059a84633d59ba38_jpg.rf.7e593dc121ec6da5b85d4945ce752bc8.jpg', width=800)

In [ ]:
!python detect.py --source ../datasets2/images/test/DJI_0009_JPG.rf.524c985e90fbfd166503bb4f1dab5348.jpg --weights runs/train/exp/weights/best.pt --conf 0.5 --save-txt

In [ ]:
Image(filename='./runs/detect/exp6/DJI_0009_JPG.rf.524c985e90fbfd166503bb4f1dab5348.jpg', width=800)

**Inference on Videos:**

In [ ]:
!python detect.py  --source ../datasets2/Video_1.mp4 --weights runs/train/exp/weights/best.pt --conf 0.25 --save-txt

In [ ]:
!python detect.py  --source ../datasets2/Video_2.mp4 --weights runs/train/exp/weights/best.pt --conf 0.25 --save-txt

In [ ]:
!python detect.py  --source ../datasets2/Video_3.mp4 --weights runs/train/exp/weights/best.pt --conf 0.25 --save-txt

In [ ]:
!python detect.py  --source ../datasets2/Video_4.mp4 --weights runs/train/exp/weights/best.pt --conf 0.75 --save-txt

References:


1.https://github.com/ultralytics/yolov5

2.https://github.com/ultralytics/yolov5/wiki/Train-Custom-Data

3.https://blog.roboflow.com/how-to-train-yolov5-on-a-custom-dataset/




